In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
# Standard Library imports
from pathlib import Path
from typing import Optional

# External imports
import fiftyone as fo
import fiftyone.brain as fob
from fiftyone.core.dataset import Dataset, load_dataset
from fiftyone.core.view import DatasetView

In [ ]:
# Path towards the root of the REORGANIZED YOLO dataset
SOURCE_PATH = Path("FILL_ME")

# Path towards the destinty folder where the dataset will be exported
TARGET_PATH = Path("FILL_ME")

FIFTYONE_DATASET_NAME = "fish_crops_RGBA"
NUM_UNIQUE = 500
TAG_TRAIN = "train"
TAG_VAL = "val"
TAG_TEST = "test"
TAG_CORRECT = "correct"
TAG_INCORRECT = "incorrect"

In [ ]:
def list_all_images(dir_path: Path):
    """ """
    suffixes = {".png", ".jpg", ".jpeg"}
    return [p.resolve() for p in dir_path.rglob("*") if p.suffix.lower() in suffixes]


def add_samples_to_dataset(
    dataset: Dataset | DatasetView,
    sources: Path | str | list[Path],
    tags: Optional[list[str]] = None,
):
    """ """

    if isinstance(sources, (Path, str)):
        sources = [Path(sources)]
    if isinstance(sources, list):
        sources = [Path(s) for s in sources]

    for source in sources:
        files = list_all_images(source)

        existing = set(dataset.values("filepath") or [])
        samples = []
        for filepath in files:
            if str(filepath.resolve()) not in existing:
                samples.append(fo.Sample(filepath=filepath, tags=tags))

        dataset.add_samples(samples)


def is_empty(directory: Path):
    return not any(directory.iterdir())


def export_yolo_classification(
    dataset: Dataset | DatasetView, splits: list[str], export_path: Path
):
    """ """
    if export_path.exists() and not is_empty(export_path):
        raise Exception("Te export dir is not empty")

    label_field = "ground_truth"

    for split in splits:
        export_dir = str(export_path / split)
        print(f"Exporting '{split}' data to '{export_dir}'")
        dataset.match_tags([split]).export(
            export_dir=export_dir,
            dataset_type=fo.types.ImageClassificationDirectoryTree,
            label_field=label_field,
        )


def add_label_to_view(view: DatasetView, new_label: str):
    """ """
    for sample in view:
        sample["ground_truth"] = fo.Classification(label=new_label)
        sample.save()
    print("Done")

In [ ]:
fo.delete_dataset(FIFTYONE_DATASET_NAME)

try:
    dataset = Dataset(FIFTYONE_DATASET_NAME, persistent=True, overwrite=False)
except ValueError:
    print("Loading dataset")
    dataset = load_dataset(FIFTYONE_DATASET_NAME, create_if_necessary=False)

In [ ]:
add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_TRAIN / TAG_CORRECT], tags=[TAG_TRAIN, TAG_CORRECT]
)
add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_TRAIN / TAG_INCORRECT], tags=[TAG_TRAIN, TAG_INCORRECT]
)

add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_VAL / TAG_CORRECT], tags=[TAG_VAL, TAG_CORRECT]
)
add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_VAL / TAG_INCORRECT], tags=[TAG_VAL, TAG_INCORRECT]
)

add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_TEST / TAG_CORRECT], tags=[TAG_TEST, TAG_CORRECT]
)
add_samples_to_dataset(
    dataset, [SOURCE_PATH / TAG_TEST / TAG_INCORRECT], tags=[TAG_TEST, TAG_INCORRECT]
)

In [ ]:
dataset.create_index("tags")

In [ ]:
# When auto=False is provided, a new App window is created only when you call session.show():
session = fo.launch_app(dataset, browser="chrome", auto=False)

# Now open http://localhost:5151

## Compute Visual Similarity Index

Build (or load) a visual similarity index over the TEST correct/incorrect samples using **CLIP** (`clip-vit-base32-torch`).

- If a dataset similarity index already exists under `brain_key_similarity`, it is loaded from cache.
- Otherwise, embed every sample with CLIP and indexes the resulting vectors, storing them under the given brain key.

The resulting `similarity_index`s are used downstream to find near-duplicate images and to select a maximally diverse subset of unique samples.

In [ ]:
test_correct_view = dataset.match_tags([TAG_TEST, TAG_CORRECT], all=True)
test_incorrect_view = dataset.match_tags([TAG_TEST, TAG_INCORRECT], all=True)

model_name = "clip-vit-base32-torch"

brain_key_correct = "similarity_correct"
# dataset.delete_brain_run(brain_key_correct)
if brain_key_correct in dataset.list_brain_runs():
    print("Loading existing similarity index (correct)")
    sim_correct = dataset.load_brain_results(brain_key_correct)
else:
    sim_correct = fob.compute_similarity(
        samples=test_correct_view, model=model_name, brain_key=brain_key_correct
    )

brain_key_incorrect = "similarity_incorrect"
# dataset.delete_brain_run(brain_key_incorrect)
if brain_key_incorrect in dataset.list_brain_runs():
    print("Loading existing similarity index (incorrect)")
    sim_incorrect = dataset.load_brain_results(brain_key_incorrect)
else:
    sim_incorrect = fob.compute_similarity(
        samples=test_incorrect_view, model=model_name, brain_key=brain_key_incorrect
    )

## Compute 2D Embedding Visualization

Project the CLIP embeddings into 2D (using UMAP by default) so they can be explored interactively in the FiftyOne App.

- Reuses the existing `similarity_index` so no re-embedding is needed.
- Results are stored under `brain_key_visualization` and can be loaded in the App's **Embeddings panel** to visually inspect clusters, outliers, and near-duplicates.

In [ ]:
# Generate a 2D visualization of the embeddings
brain_key_visualization = "visualization_test_correct"
# dataset.delete_brain_run(brain_key_visualization)
viz_test_correct_results = fob.compute_visualization(
    test_correct_view, brain_key=brain_key_visualization, similarity_index=sim_correct
)
brain_key_visualization = "visualization_test_incorrect"
# dataset.delete_brain_run(brain_key_visualization)
viz_test_incorrect_results = fob.compute_visualization(
    test_incorrect_view,
    brain_key=brain_key_visualization,
    similarity_index=sim_incorrect,
)

## Find Duplicates / Select Unique Samples

Use the similarity index to either remove near-duplicates or select a maximally diverse subset.

**Find duplicates:** `find_duplicates(thresh)` marks images as duplicates when their embedding distance falls below thresh. `duplicates_view()` updates the App to display only those samples.

**Select unique samples:** `find_unique(count)` picks the `NUM_UNIQUE` most visually spread-out samples across the embedding space. `unique_view()` updates the App to display to show only those samples.

In [ ]:
# similarity_index.find_duplicates(thresh=0.5)  # Higher thresh, more duplicates shown
# session.view = similarity_index.duplicates_view()

In [ ]:
sim_correct.find_unique(count=NUM_UNIQUE)
unique_correct_view = sim_correct.unique_view()

sim_incorrect.find_unique(count=NUM_UNIQUE)
unique_incorrect_view = sim_incorrect.unique_view()

In [ ]:
session.view = unique_correct_view
# session.view = unique_incorrect_view

## Export test filtered samples

In [ ]:
add_label_to_view(unique_correct_view, TAG_CORRECT)
add_label_to_view(unique_incorrect_view, TAG_INCORRECT)

# Combine both unique views
combined_ids = unique_correct_view.values("id") + unique_incorrect_view.values("id")
combined_view = dataset.select(combined_ids)

session.view = combined_view
export_yolo_classification(
    combined_view,
    export_path=TARGET_PATH,
    splits=[TAG_TEST],
)